In [1]:
import pandas as pd
import numpy as np
import requests
from statsmodels.tsa.stattools import coint
import matplotlib.pyplot as plt
import time

In [2]:
def get_historical_data(symbol, interval='60', num_candles=3000):
    # docs: https://bybit-exchange.github.io/docs/v5/market/kline
    url = 'https://api.bybit.com/v5/market/kline'
    all_data = []
    end = int(time.time() * 1000)

    while len(all_data) < num_candles:
        params = {
            'category': 'spot',
            'symbol': symbol,
            'interval': interval,
            'limit': 1000,
            'end': end
        }
        r = requests.get(url, params=params)
        data = r.json()['result']['list']
        if not data:
            break
        all_data.extend(data)
        end = int(data[-1][0]) - 1
        time.sleep(0.01)

    df = pd.DataFrame(all_data, columns=[
        'timestamp', 'open', 'high', 'low', 'close', 'volume', 'turnover'
    ])
    df['timestamp'] = pd.to_datetime(df['timestamp'].astype(float), unit='ms')
    df['close'] = df['close'].astype(float)
    df = df[['timestamp', 'close']].sort_values('timestamp').reset_index(drop=True)
    return df.iloc[:num_candles]

In [3]:
# docs: https://bybit-exchange.github.io/docs/v5/market/tickers
url = 'https://api.bybit.com/v5/market/tickers'
params = {'category': 'spot'}
r = requests.get(url, params=params)
tickers = [ticker['symbol'] for ticker in r.json()['result']['list']]
print(len(tickers))
print(tickers[:50])

598
['DOTUSDC', 'MERLUSDT', 'PEOPLEUSDT', 'TOKENUSDT', 'WLDUSDT', 'MNTUSDC', 'FILUSDC', 'PEPEMNT', 'VETUSDT', 'STXUSDT', 'SEIUSDC', 'CORNUSDT', 'ALGOUSDC', 'SAROSUSDT', 'PRIMEUSDT', 'STABLEUSDT', 'DYDXUSDT', 'SUSHIUSDT', 'COOKIEUSDT', 'WLFIUSDT', 'OPNUSDT', 'SOLUSDE', 'YFIUSDT', 'BELUSDT', 'AURORAUSDT', 'WUSDT', 'VANRYUSDT', 'B3USDT', 'CHIPUSDT', 'MEGAUSDT', 'SOLUSDC', 'SUSDT', 'ERAUSDT', 'HOLOUSDT', 'MOGUSDT', 'ETHMNT', 'SXTUSDT', 'GALAUSDT', 'SWELLUSDC', 'BTCAED', 'GAIBUSDT', 'KAVAUSDT', 'MANAUSDT', 'MBXUSDT', 'NAKAUSDT', 'SVLUSDT', 'MXUSDT', 'CAKEUSDT', 'LUNCUSDT', 'LUNCUSDC']


In [4]:
btc = get_historical_data('BTCUSDT')
eth = get_historical_data('ETHUSDT')
sol = get_historical_data('SOLUSDT')
btc.head()

,timestamp,close
0,2026-02-26 23:00:00,67482.8
1,2026-02-27 00:00:00,67228.7
2,2026-02-27 01:00:00,67346.2
3,2026-02-27 02:00:00,67253.8
4,2026-02-27 03:00:00,67597.3


In [5]:
data = {
    'BTC': btc,
    'ETH': eth,
    'SOL': sol,
}
major = ['BTC', 'ETH', 'SOL']

for i, a in enumerate(major):
    for j, b in enumerate(major):
        if j > i:
            _, pvalue, _ = coint(data[a]['close'], data[b]['close'])
            print(f"{a}|{b}: {pvalue:.4f}")

BTC|ETH: 0.7068
BTC|SOL: 0.9360
ETH|SOL: 0.9394


In [6]:
tickers_usdt = [t for t in tickers if t.endswith('USDT')]
print(len(tickers_usdt))
print(tickers_usdt[:50])

431
['MERLUSDT', 'PEOPLEUSDT', 'TOKENUSDT', 'WLDUSDT', 'VETUSDT', 'STXUSDT', 'CORNUSDT', 'SAROSUSDT', 'PRIMEUSDT', 'STABLEUSDT', 'DYDXUSDT', 'SUSHIUSDT', 'COOKIEUSDT', 'WLFIUSDT', 'OPNUSDT', 'YFIUSDT', 'BELUSDT', 'AURORAUSDT', 'WUSDT', 'VANRYUSDT', 'B3USDT', 'CHIPUSDT', 'MEGAUSDT', 'SUSDT', 'ERAUSDT', 'HOLOUSDT', 'MOGUSDT', 'SXTUSDT', 'GALAUSDT', 'GAIBUSDT', 'KAVAUSDT', 'MANAUSDT', 'MBXUSDT', 'NAKAUSDT', 'SVLUSDT', 'MXUSDT', 'CAKEUSDT', 'LUNCUSDT', 'REDUSDT', 'AIXBTUSDT', 'GODSUSDT', 'ROSEUSDT', 'BLASTUSDT', 'LAUSDT', 'PARTIUSDT', 'MEEUSDT', 'POLUSDT', 'VTHOUSDT', 'LINKUSDT', 'MINAUSDT']


In [7]:
print(r.json()['result']['list'][0])

{'symbol': 'DOTUSDC', 'bid1Price': '0.8381', 'bid1Size': '41.765', 'ask1Price': '0.8389', 'ask1Size': '41.765', 'lastPrice': '0.8398', 'prevPrice24h': '0.8167', 'price24hPcnt': '0.0283', 'highPrice24h': '0.8482', 'lowPrice24h': '0.807', 'turnover24h': '6496.753354', 'volume24h': '7862.783', 'usdIndexPrice': '0.837965'}


In [8]:
tickers_filt = [t_info['symbol'] for t_info in r.json()['result']['list']
                if float(t_info['turnover24h']) > 1_000_000
                and t_info['symbol'].endswith('USDT')]
print(len(tickers_filt))
print(tickers_filt[:50])

91
['WLDUSDT', 'STABLEUSDT', 'DYDXUSDT', 'LINKUSDT', 'SLXUSDT', 'KASUSDT', 'NEARUSDT', 'TIAUSDT', 'SPCXXUSDT', 'XLMUSDT', 'AAVEUSDT', 'XPLUSDT', 'SPXUSDT', 'USDCUSDT', 'ZROUSDT', 'VVVUSDT', 'FILUSDT', 'ASTERUSDT', 'USD1USDT', 'NOMUSDT', 'ICNTUSDT', 'CCUSDT', 'MANTAUSDT', 'HYPEUSDT', 'BNBUSDT', 'DOGEUSDT', 'COINXUSDT', 'BILLUSDT', 'HBARUSDT', 'XRPUSDT', 'TAIKOUSDT', 'ETHUSDT', 'MORPHOUSDT', 'USDEUSDT', 'UNIUSDT', 'CRCLXUSDT', 'CELOUSDT', 'NIGHTUSDT', 'BASEDUSDT', 'VIRTUALUSDT', 'ONDOUSDT', 'DOTUSDT', 'SOLUSDT', 'BONKUSDT', 'JUPUSDT', 'WETUSDT', 'HUSDT', 'SHIBUSDT', 'BCHUSDT', 'AVAXUSDT']
